In [19]:
import importlib
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA modules
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as gup

importlib.reload(ep)
importlib.reload(em)
importlib.reload(gup)

# Define directories
data_dir = repo_dir / "Data/GrabSample_data"
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load RI23 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

############################
# Hungerford March Thermal #
############################

Hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_martherm_fractions_df,
    hungerford_martherm_scaler,
    hungerford_martherm_pca,
    hungerford_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
        "RI23-1035", # Baseflow
        "RI23-5007", # SWLD
        "RI23-1061"  # Meltwater/Rain
    ], 
)

# 2. Extract raw endmember rows 
em_raw_subset = df[df["Sample ID"].isin(["RI23-1035", "RI23-5007", "RI23-1061"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()

stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define laboratory tracer precisions [W(C_j)]
analytical_sd = {
    "Ca_mg_L": 0.05,   # mg/L
    "Cl_mg_L": 0.15,   # mg/L
    "Si_mg_L": 0.10,   # mg/L
    "K_mg_L": 0.10,    # mg/L
    "Cu_mg_L": 0.10,   # mg/L
    "Mg_mg_L": 0.02,   # mg/L
    "dD": 0.8,         # per mil
    "d18O": 0.08,      # per mil
    "Na_mg_L": 0.05    # mg/L
}

# 5. Calculate dataset-grounded PC analytical standard deviations
calculated_pc_sd = gup.calculate_pc_analytical_sd(
    scaler=hungerford_martherm_scaler,
    pca=hungerford_martherm_pca,
    analytical_sd_dict=analytical_sd
)

print(f"Calculated PC1 analytical SD: {calculated_pc_sd['PC1']:.4f}")
print(f"Calculated PC2 analytical SD: {calculated_pc_sd['PC2']:.4f}")

# 6. Process Genereux (2022) PCA uncertainty using calculated PC precisions
confidence_levels = [0.70, 0.95]

for conf in confidence_levels:
    uncertainty_df = gup.propagate_genereux2022_pca_uncertainty(
        stream_df=stream_event_df,
        em_grouped_pca=hungerford_martherm_endmembers_df,
        em_raw=em_raw_subset,
        tracers=Hungerford_tracers,
        scaler=hungerford_martherm_scaler,
        pca=hungerford_martherm_pca,
        #pc_analytical_sd=calculated_pc_sd,  # This computes the end-member sd from the emma end-member samples
        pc_analytical_sd={'PC1': 0.70, 'PC2': 0.34}, # This is the computed end-member sd from RI25 end-members (high resolution dataset)
        confidence_level=conf
    )

    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Hungerford_Mar_Therm_PCA_Genereux2022_uncertainty_{conf_pct}pct.csv"
    uncertainty_df.to_csv(output_filename, index=False)
    #print(f"✅ Saved {conf_pct}% CI results to: {output_filename}")

Calculated PC1 analytical SD: 41.8869
Calculated PC2 analytical SD: 3.1114
